# Sentinel-2 image acquisition notebook

**Description**

This notebook downloads Sentinel-2 images via openeo library. This image is used for NDVI indices calculation and it is used as an input of deep learning model in later step.

**Input file(s)**

There is no required input files.

**Output file(s)**

1 file
- Median composite of Sentinel-2 from selected months (`sentinel2_openeo_med_4months_dresden.geotiff`)

*Selected months is summer period time in Europe (May to August 2024)*

**Requirements/Dependencies**

- To run this notebook, user needs to have an account on Copernicus Data Space Ecosystem (Registration step can be found in https://documentation.dataspace.copernicus.eu/Registration.html).
- This notebook doesn't have dependency with other notebooks.

**Additional notes**
- There might be a possibility to get the timeout when running **download** function. The suggestion is re-running the cells again or restart the kernel and run all code again.
- It could take up 2-5 minutes to complete the download step

# 1. Import libraries

In [1]:
import openeo
import os
import yaml

In [2]:
# Load config
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# 2. Connect to openeo library

In [3]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
# connection = openeo.connect("openeofed.dataspace.copernicus.eu")

In [4]:
connection

<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with NullAuth>

In [5]:
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

# 3. Load Sentinel-2 collection

bbox of Dresden, Germany
- [13.5793237, 50.974937 , 13.9660626, 51.1777202] 

In [6]:
sentinel2_l2a_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent=config["data"]["spatial_extent"],
    temporal_extent=config["data"]["temporal_extent"],
    max_cloud_cover=config["data"]["max_cloud_cover"],
    bands=[
        "B01",
        "B02",
        "B03",
        "B04",
        "B05",
        "B06",
        "B07",
        "B08",
        "B8A",
        "B09",
        "B11",
        "B12",
        "SCL",
    ],
)

sentinel2_l1c_cube = connection.load_collection(
    "SENTINEL2_L1C",
    spatial_extent=config["data"]["spatial_extent"],
    temporal_extent=config["data"]["temporal_extent"],
    max_cloud_cover=config["data"]["max_cloud_cover"],
    bands=[
        "B01",
        "B02",
        "B03",
        "B04",
        "B05",
        "B06",
        "B07",
        "B08",
        "B8A",
        "B09",
        "B10",
        "B11",
        "B12",
    ],
)

In [7]:
sentinel2_l2a_cube.print_json()

{
  "process_graph": {
    "loadcollection1": {
      "process_id": "load_collection",
      "arguments": {
        "bands": [
          "B01",
          "B02",
          "B03",
          "B04",
          "B05",
          "B06",
          "B07",
          "B08",
          "B8A",
          "B09",
          "B11",
          "B12",
          "SCL"
        ],
        "id": "SENTINEL2_L2A",
        "properties": {
          "eo:cloud_cover": {
            "process_graph": {
              "lte1": {
                "process_id": "lte",
                "arguments": {
                  "x": {
                    "from_parameter": "value"
                  },
                  "y": 5
                },
                "result": true
              }
            }
          }
        },
        "spatial_extent": {
          "west": 13.5793237,
          "south": 50.974937,
          "east": 13.9660626,
          "north": 51.1777202
        },
        "temporal_extent": [
          "2024-05-01",
  

In [8]:
sentinel2_l1c_cube.print_json()

{
  "process_graph": {
    "loadcollection1": {
      "process_id": "load_collection",
      "arguments": {
        "bands": [
          "B01",
          "B02",
          "B03",
          "B04",
          "B05",
          "B06",
          "B07",
          "B08",
          "B8A",
          "B09",
          "B10",
          "B11",
          "B12"
        ],
        "id": "SENTINEL2_L1C",
        "properties": {
          "eo:cloud_cover": {
            "process_graph": {
              "lte1": {
                "process_id": "lte",
                "arguments": {
                  "x": {
                    "from_parameter": "value"
                  },
                  "y": 5
                },
                "result": true
              }
            }
          }
        },
        "spatial_extent": {
          "west": 13.5793237,
          "south": 50.974937,
          "east": 13.9660626,
          "north": 51.1777202
        },
        "temporal_extent": [
          "2024-05-01",
  

# 4. Apply cloud masking

For Sentinel-2 L2A data, there is *scence classification* layer (band name *SCL*) generated by the Sen2Cor algorithm. This layer information is used to create binary cloud mask.

Steps to apply cloud mask:
1. Create binary cloud mask from the SCL values 3 (cloud shadows), 8 (cloud medium probability) and 9 (cloud high probability)
2. Apply cloud masking on sentinel2_l1c_cube

In [9]:
scl_band = sentinel2_l2a_cube.band("SCL")
cloud_mask = (scl_band == 3) | (scl_band == 8) | (scl_band == 9) | (scl_band == 10)

In [10]:
cube_masked = sentinel2_l1c_cube.mask(cloud_mask)

# 5. Compute median over the collection (multiple images)

In [11]:
sentinel2_l1c_cube_med = cube_masked.reduce_dimension(dimension="t", reducer="median")

# 6. Download median composite image

It could take up 2-5 minutes to complete the download step

In [12]:
output_path = config["paths"]["raw_sentinel_path"]

if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))
    
sentinel2_l1c_cube_med.download(output_path,format="GTiff")